# Data Loading

Этот notebook использует базовый package interface `PolymarketDataset`. Здесь нет research-specific panels, feature engineering или benchmark logic: только загрузка core dataset из локальной SQLite базы и сохранение базовых артефактов в `running_artefacts`.

## Что делает этот notebook

1. Импортирует `PolymarketDataset`.
2. Создаёт dataset object с явными параметрами выборки.
3. Загружает базовые таблицы `markets` и `probabilities`.
4. Показывает краткое summary и несколько первых строк.
5. Сохраняет dataset в `research_notebooks/running_artefacts`.

Это должен быть самый простой и читаемый entry point в данные.

## Импорт и конфигурация

Проект теперь установлен в env через editable install, поэтому notebook может делать обычные package imports без ручного `sys.path` bootstrap. Ниже задаются только базовые параметры датасета: какие домены брать, сколько рынков на домен загружать и какой минимальный объём history считать достаточным.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from polymarket_research import PolymarketDataset

In [6]:
DOMAINS = ('politics', 'geopolitics', 'technology', 'finance_economy')
MAX_MARKETS_PER_DOMAIN = 250
MIN_PROBABILITY_ROWS = 288
ARTEFACT_DIR = Path.cwd().resolve() / 'running_artefacts'
ARTEFACT_DIR.mkdir(parents=True, exist_ok=True)

dataset = PolymarketDataset(
    domains=DOMAINS,
    max_markets_per_domain=MAX_MARKETS_PER_DOMAIN,
    min_probability_rows=MIN_PROBABILITY_ROWS,
)

## Явные assumptions

- Используем только полноценно загруженные домены `politics`, `geopolitics`, `technology`, `finance_economy`.
- Не используем Polymarket `crypto` domain.
- Ограничиваемся максимум `120` рынками на домен.
- Берём только рынки с не менее чем `288` probability rows.

Это не scientific claim, а просто reproducible starting point для дальнейшей работы.

## Загрузка базового датасета

Ниже один основной вызов: `dataset.load()`. После него в объекте лежат две таблицы:

- `dataset.markets`
- `dataset.probabilities`

Именно их мы дальше будем использовать как сырой вход для всех research notebooks.

In [7]:
dataset.load()
summary = dataset.summary()
display(summary)

In [8]:
dataset.probabilities

## Что именно загрузилось

Сначала полезно посмотреть на покрытие по доменам, затем на несколько строк из обеих таблиц.

Важные колонки в `markets`:
- `market_id`
- `question`
- `primary_domain`
- `trade_rows`
- `probability_rows`

Важные колонки в `probabilities`:
- `market_id`
- `timestamp_utc`
- `yes_probability`
- `observed_trade`
- `trade_count`
- `total_size`

In [9]:
markets_by_domain = dataset.markets.groupby('primary_domain', dropna=False).agg(
    markets=('market_id', 'nunique'),
    mean_probability_rows=('probability_rows', 'mean'),
    mean_trade_rows=('trade_rows', 'mean'),
).reset_index().sort_values('markets', ascending=False)

display(markets_by_domain)
display(dataset.markets.head(10))
display(dataset.probabilities.head(10))

## Сохранение в running_artefacts

Теперь сохраняем только базовый dataset: `markets.parquet` и `probabilities.parquet`. Все промежуточные представления лежат в `research_notebooks/running_artefacts`, а сам notebook остаётся на уровне `research_notebooks`.

In [10]:
manifest = dataset.save(ARTEFACT_DIR)
display(manifest)
print('Saved basic dataset to', ARTEFACT_DIR)

Saved basic dataset to /Users/sneddy/research/polymarket_research/research_notebooks/running_artefacts
